# Liperty VSR Training Notebook (Parallel & GDrive Persistent)
This version of the notebook is optimized for **speed** (simultaneous downloads) and **persistence** (Google Drive storage).

**Features:**
- Uses `aria2c` for high-speed parallel downloads.
- Persistent storage on Google Drive (`MyDrive/LipertyData`).
- Automated setup and LoRA initialization.

In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 2. Define Persistence Paths
import os
gdrive_data_root = '/content/drive/MyDrive/LipertyData'
!mkdir -p {gdrive_data_root}

# 3. Clone Repository
repo_dir = '/content/Liperty'
if not os.path.exists(repo_dir):
    !git clone https://github.com/HereLiesAz/Liperty.git {repo_dir}

%cd {repo_dir}

# 4. Symlink GDrive Data to the Repo
!ln -snf {gdrive_data_root} /content/Liperty/data

# 5. Install Dependencies (including aria2 for speed)
!apt-get install -y aria2
!./setup_libs.sh
!pip install datasets transformers mediapipe opencv-python onnx onnx-tf tensorflow torch

## High-Speed Parallel Data Preparation
The following cell downloads all specified datasets simultaneously to your Google Drive.

In [ ]:
import os

# Prepare directories
!mkdir -p data/LRS2-2Mix data/VVAD-LRS3

# [1] Setup Kaggle for VVAD
if os.path.exists("/content/drive/MyDrive/kaggle.json"):
    !mkdir -p ~/.kaggle
    !cp /content/drive/MyDrive/kaggle.json ~/.kaggle/
    !chmod 600 ~/.kaggle/kaggle.json

# [2] Start Parallel Downloads
print("Starting simultaneous downloads for LRS2-2Mix and VVAD-LRS3...")

# Download LRS2-2Mix in background via aria2c
lrs2_url = "https://huggingface.co/datasets/JusperLee/LRS2-2Mix/resolve/main/lrs2.tar.gz"
if not os.path.exists("data/LRS2-2Mix/lrs2.tar.gz"):
    !aria2c -x 16 -s 16 -o data/LRS2-2Mix/lrs2.tar.gz {lrs2_url} &
else:
    print("LRS2-2Mix already exists.")

# Download VVAD-LRS3 in background via kaggle cli
if not os.path.exists("data/VVAD-LRS3/vvadlrs3.zip") and os.path.exists("/root/.kaggle/kaggle.json"):
    !kaggle datasets download -d adrianlubitz/vvadlrs3 -p data/VVAD-LRS3 &
else:
    print("VVAD-LRS3 already exists or kaggle.json missing.")

print("Downloads started in background. Waiting for all processes to finish...")
!wait

print("\nExtracting files...")
# Extract (parallel-ish via consecutive &)
if os.path.exists("data/LRS2-2Mix/lrs2.tar.gz"):
    !tar -xzf data/LRS2-2Mix/lrs2.tar.gz -C data/LRS2-2Mix --strip-components=1 &
if os.path.exists("data/VVAD-LRS3/vvadlrs3.zip"):
    !unzip -q data/VVAD-LRS3/vvadlrs3.zip -d data/VVAD-LRS3 &

!wait
print("All datasets ready on Google Drive!")

## Training & Fine-Tuning

In [ ]:
!mkdir -p data/checkpoints
!python tools/create_trainable_model.py
print("Ready for training.")

## Export to TFLite

In [ ]:
!python tools/convert_vallr.py